In [3]:
import pennylane as qml
import numpy as np

# --- Configuration ---
n_qubits = 3
wires = range(n_qubits)
dev = qml.device('default.qubit', wires=n_qubits)

# --- Define the Oracle (Uf) ---
# Marks state |101>
def oracle():
    # Flips the sign of the amplitude for state |101>
    # PennyLane's FlipSign is a direct implementation of Uf = I - 2|w><w|
    qml.FlipSign([1, 0, 1], wires=wires)

# --- Define the Diffuser (W) ---
# Implements: H -> (2|0><0| - I) -> H
def diffuser():
    # H on all wires
    for w in wires:
        qml.Hadamard(wires=w)

    # Reflection about |0>: 2|0><0| - I
    # Apply X to all wires
    for w in wires:
        qml.PauliX(wires=w)
    
    # Multi-controlled Z construction
    qml.Hadamard(wires=wires[-1])
    
    # --- FIXED LINE BELOW ---
    # We specify control_wires (first n-1 qubits) and wires (target qubit) separately
    qml.MultiControlledX(wires=wires)
    
    qml.Hadamard(wires=wires[-1])
    
    # Apply X to all wires
    for w in wires:
        qml.PauliX(wires=w)

    # H on all wires
    for w in wires:
        qml.Hadamard(wires=w)

@qml.qnode(dev)
def grover_circuit():
    # 1. Initialization: H on all qubits
    for w in wires:
        qml.Hadamard(wires=w)

    # 2. Iteration (Oracle + Diffuser)
    num_iterations = 2
    for _ in range(num_iterations):
        oracle()
        diffuser()

    return qml.probs(wires=wires)

# --- Run ---
probs = grover_circuit()

print(f"\nPennyLane Probabilities for 3-qubit Grover (Target |101>):")
# Index 5 corresponds to binary 101
for i, prob in enumerate(probs):
    print(f"State |{i:03b}>: {prob:.4f}")


PennyLane Probabilities for 3-qubit Grover (Target |101>):
State |000>: 0.0078
State |001>: 0.0078
State |010>: 0.0078
State |011>: 0.0078
State |100>: 0.0078
State |101>: 0.9453
State |110>: 0.0078
State |111>: 0.0078
